---
## Signature Detection Evaluation

Evaluates the model's ability to detect handwritten signatures using:
- **Positive samples**: SigDetectVerifyFlow dataset (documents WITH signatures)
- **Negative samples**: CORD receipts (documents WITHOUT signatures)

Metrics: Accuracy, Precision, Recall, F1

In [ ]:
from app.evaluate import evaluate_signature_single, evaluate_signature_batch


In [ ]:
from datasets import load_dataset
import numpy as np

N_SIG = 10  # images per class (10 positive + 10 negative = 20 total)

# Positive: documents with signatures
sig_ds = load_dataset("Mels22/SigDetectVerifyFlow", split="test")
pos_indices = np.linspace(0, len(sig_ds) - 1, N_SIG, dtype=int).tolist()
pos_images = [sig_ds[i]["document"].convert("RGB") for i in pos_indices]

# Negative: CORD receipts (no signatures)
cord = load_dataset("naver-clova-ix/cord-v2", split="test")
neg_indices = np.linspace(0, len(cord) - 1, N_SIG, dtype=int).tolist()
neg_images = [cord[i]["image"].convert("RGB") for i in neg_indices]

all_images = pos_images + neg_images
all_labels = [True] * N_SIG + [False] * N_SIG

print(f"Loaded {len(all_images)} images ({N_SIG} with signatures, {N_SIG} without)")

# Run extraction
predictions = []
for i, img in enumerate(all_images):
    result = extractor.extract(img)  # unified prompt
    predictions.append(result["parsed_json"])
    label = "+" if all_labels[i] else "-"
    print(f"  [{label}] Image {i+1}/{len(all_images)}: JSON valid={result['json_valid']}")

# Evaluate
results = evaluate_signature_batch(predictions, all_labels)

print(f"\n{'='*50}")
print(f"  SIGNATURE DETECTION RESULTS")
print(f"{'='*50}")
print(f"  Accuracy:  {results['accuracy']:.3f}")
print(f"  Precision: {results['precision']:.3f}")
print(f"  Recall:    {results['recall']:.3f}")
print(f"  F1:        {results['f1']:.3f}")
print(f"  TP={results['tp']}  FP={results['fp']}  FN={results['fn']}  TN={results['tn']}")